In [1]:
!pip install pymupdf


In [2]:
import fitz  # PyMuPDF
import os

# Set your folder path here
input_folder = "PDF sources"
output_folder = "sources"
os.makedirs(output_folder, exist_ok=True)

# Process all PDF files in the folder
for filename in os.listdir(input_folder):
    if filename.endswith(".pdf"):
        filepath = os.path.join(input_folder, filename)
        doc = fitz.open(filepath)

        toc = doc.get_toc()
        if not toc:
            print(f"No TOC found in {filename}. Skipping.")
            continue

        print(f"Processing {filename} with {len(toc)} TOC entries...")

        for i, entry in enumerate(toc):
            level, title, start_page = entry
            end_page = toc[i + 1][2] - 1 if i + 1 < len(toc) else doc.page_count - 1
            chapter_text = ''
            for page_num in range(start_page - 1, end_page + 1):
                chapter_text += doc[page_num].get_text()

            # Clean and name output file
            safe_title = f"{i+1:02d}_{title.replace('/', '_').replace(' ', '_')[:50]}"
            out_filename = f"{os.path.splitext(filename)[0]}__{safe_title}.txt"
            with open(os.path.join(output_folder, out_filename), "w", encoding="utf-8") as f:
                f.write(chapter_text)

        print(f"✓ Finished splitting {filename} into chapters.")



No TOC found in Nimako et al_2020_Een rapport met betrekking tot het onderzoeksproject over de periode voor de.pdf. Skipping.
No TOC found in ZWART_MANIFEST.pdf. Skipping.


In [1]:
import fitz
import os
import re

# Set your folder path here
input_folder = "PDF sources"
output_folder = "sources"
os.makedirs(output_folder, exist_ok=True)

# Regex pattern: capture chapter title and page number
toc_line_pattern = re.compile(r'^(.*?)\.{2,}\s*(\d+)$')

for filename in os.listdir(input_folder):
    if not filename.endswith(".pdf"):
        continue

    filepath = os.path.join(input_folder, filename)
    doc = fitz.open(filepath)
    print(f"📘 Processing {filename}")

    # Step 1: Look for a page with TOC structure in first 10 pages
    toc_entries = []
    for i in range(min(10, doc.page_count)):
        text = doc[i].get_text()
        for line in text.splitlines():
            match = toc_line_pattern.match(line.strip())
            if match:
                title, page_num = match.groups()
                toc_entries.append((title.strip(), int(page_num)))

    if not toc_entries:
        print(f"⚠️ No TOC lines detected visually in {filename}")
        continue

    # Step 2: Extract chapter content
    for i, (title, start_page) in enumerate(toc_entries):
        end_page = toc_entries[i + 1][1] - 1 if i + 1 < len(toc_entries) else doc.page_count
        content = ''
        for p in range(start_page - 1, end_page):
            content += doc[p].get_text()

        safe_title = f"{i+1:02d}_{title.replace('/', '_').replace(' ', '_')[:50]}"
        out_filename = f"{os.path.splitext(filename)[0]}__{safe_title}.txt"
        with open(os.path.join(output_folder, out_filename), "w", encoding="utf-8") as f:
            f.write(content)

    print(f"✅ Saved {len(toc_entries)} chapters for {filename}")


📘 Processing Nimako et al_2020_Een rapport met betrekking tot het onderzoeksproject over de periode voor de.pdf
⚠️ No TOC lines detected visually in Nimako et al_2020_Een rapport met betrekking tot het onderzoeksproject over de periode voor de.pdf
📘 Processing ZWART_MANIFEST.pdf
⚠️ No TOC lines detected visually in ZWART_MANIFEST.pdf


In [ ]:
import fitz  # PyMuPDF
import os
import re

input_folder = "PDF sources"
output_folder = "sources"
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if not filename.endswith(".pdf"):
        continue

    pdf_path = os.path.join(input_folder, filename)
    doc = fitz.open(pdf_path)
    print(f"📄 Processing: {filename}")
# Step 1: Try embedded TOC
toc = doc.get_toc()

if toc:
    print("📘 Using embedded TOC")
    toc_entries = [(entry[1], entry[2]) for entry in toc]  # (title, page)
else:
    # Step 2: Look for visual TOC in first 5 pages
    print("📘 No embedded TOC found — trying visual TOC")
    toc_text_lines = []
    for i in range(min(5, doc.page_count)):
        toc_text_lines.extend(doc[i].get_text().splitlines())

    # Pattern 1: Match "Title ........ 14"
    visual_pattern_1 = re.compile(r"^(.*?)\.{2,}\s*(\d{1,3})$")
    visual_matches_1 = []
    for line in toc_text_lines:
        match = visual_pattern_1.match(line.strip())
        if match:
            title, page = match.groups()
            visual_matches_1.append((title.strip(), int(page)))

    if visual_matches_1:
        print("📘 Using visual TOC (dots + page numbers)")
        toc_entries = visual_matches_1

    else:
        # Step 3: Last resort — scan for internal headers like "1> EDUCATIE"
        print("📘 Trying section markers like '1> EDUCATIE' inside body")
        section_titles = [
            "1> EDUCATIE",
            "2> ZORG EN WELZIJN",
            "3> HUISVESTING, WONEN EN OMGEVING",
            "4> ARBEIDSMARKT, WERK EN INKOMEN",
            "5> SOCIAAL-ECONOMISCHE GELIJKHEID",
            "6> POLITIEK EN OVERHEID",
            "7> (WETS)HANDHAVERS, JUSTITIE EN VEILIGHEID",
            "8> HERVORMING ANTIDISCRIMINATIEVOORZIENINGEN (ADV)",
            "9> MEDIA",
            "10> KUNST EN CULTUUR",
            "11> SPORT",
            "12> NAZORGPAKKET NAZATEN"
        ]

        page_starts = []
        for i, page in enumerate(doc):
            txt = page.get_text()
            for title in section_titles:
                if title in txt:
                    page_starts.append((title, i))

        if not page_starts:
            raise ValueError("❌ No recognizable TOC format found.")

        # Convert to same format as TOC: (title, page number + 1)
        toc_entries = [(title, page + 1) for title, page in sorted(page_starts, key=lambda x: x[1])]

# --- Extract sections based on toc_entries ---
for i, (title, start_page) in enumerate(toc_entries):
    end_page = toc_entries[i + 1][1] - 1 if i + 1 < len(toc_entries) else doc.page_count
    content = ''
    for p in range(start_page - 1, end_page):
        content += doc[p].get_text()

    safe_title = f"{i+1:02d}_{title.replace('/', '_').replace('>', '').replace(' ', '_')[:50]}"
    with open(os.path.join(output_folder, f"{safe_title}.txt"), "w", encoding="utf-8") as f:
        f.write(content)

print(f"✅ Extracted {len(toc_entries)} sections to: {output_folder}")


📄 Processing: Nimako et al_2020_Een rapport met betrekking tot het onderzoeksproject over de periode voor de.pdf
📄 Processing: ZWART_MANIFEST.pdf
📘 No embedded TOC found — trying visual TOC
📘 Trying section markers like '1> EDUCATIE' inside body


TypeError: 'in <string>' requires string as left operand, not tuple

In [8]:
doc[page].get_text()


AssertionError: Invalid item number: i=page 0 of PDF sources\ZWART_MANIFEST.pdf.

In [9]:
for i in range(3):
    print(f"Page {i+1} preview:")
    print(repr(doc[i].get_text()))


Page 1 preview:
'1\nZWART MANIFEST \nMANIFEST TER BESTRIJDING VAN INSTITUTIONEEL ANTI-ZWART RACISME \nEN TER BEVORDERING VAN ZWARTE EMANCIPATIE IN NEDERLAND\nWWW.ZWARTMANIFEST.NL\nLANCERINGSDATUM 25 MAART 2021 - VERSIE 1\n'
Page 2 preview:
'2\nZWART MANIFEST \nMANIFEST TER BESTRIJDING VAN INSTITUTIONEEL ANTI-ZWART RACISME \nEN TER BEVORDERING VAN ZWARTE EMANCIPATIE IN NEDERLAND\nArtikel 1 van de Grondwet stelt dat iedereen in Nederland in gelijke gevallen, gelijk wordt behandeld. \nHet zijn woorden die iedere Nederlander de garantie moeten bieden op een gelijkwaardig bestaan. \nDe realiteit is anders. In Nederland is sprake van een geracialiseerde orde. Het is onaanvaardbaar en \nonhoudbaar dat in deze samenleving ongelijkheid en onrechtvaardigheid op basis van ‘ras’ aan de \norde van de dag zijn. Dit manifest is opgesteld om verdere toename van die ongelijkheid en onrecht-\nvaardigheid tegen te gaan. \nAANLEIDING\nHet (anti-zwart) racisme in de samenleving is een gevolg van het koloni

In [11]:
import fitz  # PyMuPDF
import os
import re

# Set your folder paths
input_folder = "PDF sources"
output_folder = "sources"
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if not filename.endswith(".pdf"):
        continue

    pdf_path = os.path.join(input_folder, filename)
    doc = fitz.open(pdf_path)
    print(f"\n📄 Processing: {filename}")

    # Step 1: Try embedded TOC
    toc = doc.get_toc()
    toc_entries = []

    if toc:
        print("📘 Using embedded TOC")
        toc_entries = [(entry[1], entry[2]) for entry in toc]

    else:
        # Step 2: Try visual TOC (e.g., "Hoofdstuk Twee .......... 14")
        print("📘 No embedded TOC — trying visual TOC (dotted lines)")

        toc_text_lines = []
        for i in range(min(5, doc.page_count)):
            toc_text_lines.extend(doc[i].get_text().splitlines())

        visual_pattern_1 = re.compile(r"^(.*?)\.{2,}\s*(\d{1,3})$")
        visual_matches = []
        for line in toc_text_lines:
            match = visual_pattern_1.match(line.strip())
            if match:
                title, page = match.groups()
                visual_matches.append((title.strip(), int(page)))

        if visual_matches:
            print("📘 Using visual TOC (dots + page numbers)")
            toc_entries = visual_matches

        else:
            # Step 3: Scan full PDF for numbered headers like "1> EDUCATIE"
            print("📘 Trying section headers inside body like '1> EDUCATIE'")
            page_starts = []
            header_pattern = re.compile(r"^(\d{1,2}> .+)", re.MULTILINE)

            for i, page in enumerate(doc):
                text = doc[i].get_text()
                matches = header_pattern.findall(text)
                for match in matches:
                    page_starts.append((match.strip(), i))

            if not page_starts:
                print("❌ No recognizable TOC format found. Skipping file.")
                continue

            toc_entries = [(title, page + 1) for title, page in sorted(page_starts, key=lambda x: x[1])]

    # --- Extract content by ranges ---
    print("📎 Found section starts:")
    for title, page in toc_entries:
        print(f"  - {title} (page {page})")

    for i, (title, start_page) in enumerate(toc_entries):
        end_page = toc_entries[i + 1][1] - 1 if i + 1 < len(toc_entries) else doc.page_count
        content = ''

        for p in range(start_page - 1, end_page):
            page_text = doc[p].get_text()
            if not page_text.strip():
                blocks = doc[p].get_text("blocks")
                page_text = "\n".join(block[4] for block in blocks if block[4].strip())
            content += page_text + "\n"

        if content.strip():
            safe_title = f"{i+1:02d}_{title.replace('/', '_').replace('>', '').replace(' ', '_')[:50]}"
            out_path = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}__{safe_title}.txt")
            with open(out_path, "w", encoding="utf-8") as f:
                f.write(content)
        else:
            print(f"⚠️ No content found for section: {title} (pages {start_page}–{end_page})")

print("\n✅ Done processing all PDFs.")



📄 Processing: Nimako et al_2020_Een rapport met betrekking tot het onderzoeksproject over de periode voor de.pdf
📘 No embedded TOC — trying visual TOC (dotted lines)
📘 Trying section headers inside body like '1> EDUCATIE'
❌ No recognizable TOC format found. Skipping file.

📄 Processing: ZWART_MANIFEST.pdf
📘 No embedded TOC — trying visual TOC (dotted lines)
📘 Trying section headers inside body like '1> EDUCATIE'
📎 Found section starts:
  - 1> EDUCATIE (page 4)
  - 2> ZORG EN WELZIJN (page 4)
  - 3> HUISVESTING, WONEN EN OMGEVING (page 4)
  - 4> ARBEIDSMARKT, WERK EN INKOMEN (page 4)
  - 5> SOCIAAL-ECONOMISCHE GELIJKHEID (page 4)
  - 6> POLITIEK EN OVERHEID (page 4)
  - 7> (WETS)HANDHAVERS, JUSTITIE EN VEILIGHEID (page 4)
  - 8> HERVORMING ANTIDISCRIMINATIEVOORZIENINGEN (ADV) (page 4)
  - 9> MEDIA (page 4)
  - 10> KUNST EN CULTUUR (page 4)
  - 11> SPORT (page 4)
  - 12> NAZORGPAKKET NAZATEN (page 4)
  - 10> (page 33)
⚠️ No content found for section: 1> EDUCATIE (pages 4–3)
⚠️ No conten

In [ ]:
import fitz  # PyMuPDF
import os
import re

# Folder setup
input_folder = "PDF sources"
output_folder = "sources"
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if not filename.endswith(".pdf"):
        continue

    pdf_path = os.path.join(input_folder, filename)
    doc = fitz.open(pdf_path)
    print(f"\n📄 Processing: {filename}")

    toc_entries = []

    # STEP 1: Try embedded TOC
    embedded = doc.get_toc()
    if embedded:
        print("📘 Using embedded TOC")
        toc_entries = [(entry[1], entry[2]) for entry in embedded]

    else:
        # STEP 2: Try visual TOC (dots + page numbers) in first 5 pages
        print("📘 Trying visual TOC with dotted lines")
        toc_text_lines = []
        for i in range(min(5, doc.page_count)):
            toc_text_lines.extend(doc[i].get_text().splitlines())

        visual_pattern = re.compile(r"^(.*?)\.{2,}\s*(\d{1,3})$")
        visual_matches = []
        for line in toc_text_lines:
            match = visual_pattern.match(line.strip())
            if match:
                title, page = match.groups()
                visual_matches.append((title.strip(), int(page)))

        if visual_matches:
            print("📘 Using visual TOC (dotted)")
            toc_entries = visual_matches

        else:
            # STEP 3: Regex search inside full doc for domain/chapter headers
            print("📘 Using regex detection for domain headers and chapters")
            pattern_domain = re.compile(r"^(\d{1,2})>\s+([A-Z& ]+)$", re.MULTILINE)
            pattern_chapter = re.compile(r"^Hoofdstuk\s+([A-Z][a-z]+|\w+)\s*$", re.MULTILINE)

            found = []

            for i, page in enumerate(doc):
                text = doc[i].get_text()

                # Domain headers like "2> ZORG & WELZIJN"
                for number, title in pattern_domain.findall(text):
                    found.append((f"{number}> {title.strip()}", i + 1))

                # Chapters like "Hoofdstuk Twee" followed by title
                lines = text.splitlines()
                for j, line in enumerate(lines):
                    if pattern_chapter.match(line.strip()):
                        if j + 1 < len(lines):
                            real_title = lines[j + 1].strip()
                            if real_title:
                                found.append((real_title, i + 1))

            if not found:
                print("❌ No recognizable TOC format found. Skipping file.")
                continue

           
            # Filter: remove TOC-page matches and duplicates
            seen_titles = set()
            filtered_entries = []

            for title, page in sorted(found, key=lambda x: x[1]):
                # Only include real headers after page 4 (avoid visual TOC on page 1–4)
                if page > 4 and title not in seen_titles:
                    seen_titles.add(title)
                    filtered_entries.append((title, page))

            toc_entries = filtered_entries





    # --- Extract sections ---
    print("📎 Found section starts:")
    for title, page in toc_entries:
        print(f"  - {title} (page {page})")

    for i, (title, start_page) in enumerate(toc_entries):
        end_page = toc_entries[i + 1][1] - 1 if i + 1 < len(toc_entries) else doc.page_count
        content = ""

        for p in range(start_page - 1, end_page):
            page_text = doc[p].get_text()
            if not page_text.strip():
                blocks = doc[p].get_text("blocks")
                page_text = "\n".join(block[4] for block in blocks if block[4].strip())
            content += page_text + "\n"

        if content.strip():
            safe_title = f"{i+1:02d}_{title.replace('/', '_').replace('>', '').replace(' ', '_')[:50]}"
            out_path = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}__{safe_title}.txt")
            with open(out_path, "w", encoding="utf-8") as f:
                f.write(content)
        else:
            print(f"⚠️ No content found for: {title} (page {start_page})")

print("\n✅ All PDFs processed.")



📄 Processing: Nimako et al_2020_Een rapport met betrekking tot het onderzoeksproject over de periode voor de.pdf
📘 Trying visual TOC with dotted lines
📘 Using regex detection for domain headers and chapters
📎 Found section starts:

📄 Processing: ZWART_MANIFEST.pdf
📘 Trying visual TOC with dotted lines
📘 Using regex detection for domain headers and chapters
📎 Found section starts:

✅ All PDFs processed.


In [1]:
import fitz  # PyMuPDF
import os

# Folder setup
input_folder = "PDF sources"
output_folder = "Sources"
os.makedirs(output_folder, exist_ok=True)

# Segment size in pages
segment_size = 3

# Process each PDF
for filename in os.listdir(input_folder):
    if not filename.endswith(".pdf"):
        continue

    pdf_path = os.path.join(input_folder, filename)
    doc = fitz.open(pdf_path)
    base = os.path.splitext(filename)[0]

    print(f"📄 Processing {filename} ({doc.page_count} pages)")

    segment_count = 0
    for i in range(0, doc.page_count, segment_size):
        text = ''
        for j in range(i, min(i + segment_size, doc.page_count)):
            page_text = doc[j].get_text()
            if not page_text.strip():
                blocks = doc[j].get_text("blocks")
                page_text = "\n".join(block[4] for block in blocks if block[4].strip())
            text += page_text + "\n"

        if text.strip():
            out_name = f"{base}__segment_{segment_count+1:02d}.txt"
            out_path = os.path.join(output_folder, out_name)
            with open(out_path, "w", encoding="utf-8") as f:
                f.write(text.strip())
            segment_count += 1

    print(f"✅ Saved {segment_count} segments for {filename}")


📄 Processing Allen e.a. - 2023 - Staat en slavernij het Nederlandse koloniale slavernijverleden en zijn doorwerkingen.pdf (480 pages)
✅ Saved 160 segments for Allen e.a. - 2023 - Staat en slavernij het Nederlandse koloniale slavernijverleden en zijn doorwerkingen.pdf
📄 Processing Jouwe e.a. - Slavernij en de stad Utrecht.pdf (328 pages)
✅ Saved 110 segments for Jouwe e.a. - Slavernij en de stad Utrecht.pdf
📄 Processing ketenen van het verleden.pdf (272 pages)
✅ Saved 90 segments for ketenen van het verleden.pdf
📄 Processing Nimako et al_2020_Een rapport met betrekking tot het onderzoeksproject over de periode voor de.pdf (166 pages)
✅ Saved 55 segments for Nimako et al_2020_Een rapport met betrekking tot het onderzoeksproject over de periode voor de.pdf
📄 Processing ZWART_MANIFEST.pdf (48 pages)
✅ Saved 16 segments for ZWART_MANIFEST.pdf
